[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/06-evaluare-optimizare/extra-06-exemplu-clasificator-svm.ipynb)

In [1]:
# Configurare curs SEED: rulați această celulă prima.
# În Colab descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    %cd /content/SEED/cursuri/06-evaluare-optimizare
sys.path.insert(0, os.path.abspath("../.."))

In [2]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn import metrics


In [3]:
df_heart = pd.read_csv('./date/Heart.csv',header = 0).iloc[:,1:]
df_heart = df_heart.dropna(how = 'any')
df_heart['AHD']= df_heart["AHD"].apply(lambda x: 1 if x == 'Yes' else 0)
df_heart = pd.get_dummies(df_heart, columns = ['ChestPain','Thal'])
df_heart.info()

<class 'pandas.DataFrame'>
Index: 297 entries, 0 to 301
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     297 non-null    int64  
 1   Sex                     297 non-null    int64  
 2   RestBP                  297 non-null    int64  
 3   Chol                    297 non-null    int64  
 4   Fbs                     297 non-null    int64  
 5   RestECG                 297 non-null    int64  
 6   MaxHR                   297 non-null    int64  
 7   ExAng                   297 non-null    int64  
 8   Oldpeak                 297 non-null    float64
 9   Slope                   297 non-null    int64  
 10  Ca                      297 non-null    float64
 11  AHD                     297 non-null    int64  
 12  ChestPain_asymptomatic  297 non-null    bool   
 13  ChestPain_nonanginal    297 non-null    bool   
 14  ChestPain_nontypical    297 non-null    bool   
 15  Chest

In [4]:
X = df_heart[[c for c in df_heart.columns if c != 'AHD']]
y = df_heart['AHD']

## Exemplu de antrenare clasificator SVM

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def antrenament_scikitlearn_svmclsf(X,y):
    
    # instațiem un model
    svmclsf = make_pipeline(StandardScaler(), svm.SVC())  # SVM are nevoie de date scalate
    
    # setăm parametrii pe care vrem să îi încercăm
    param_grid ={'svc__kernel':['linear','poly','rbf'],
             'svc__C':[ 0.5 ,0.1, 1, 10, 100, 1000]}
    
    # folosim  antrenare cu validare încrucișată
    CV_svmclsf = GridSearchCV(
                        estimator=svmclsf, 
                          param_grid=param_grid, 
                          scoring = 'accuracy', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 5, # validare incrucistata cu impartire in 5 seturi egale de date
                          n_jobs = 6, # nr de sarcini paralele execute
                         error_score='raise')
    # pasul de antrenare
    CV_svmclsf.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_svmclsf.best_params_, CV_svmclsf.best_estimator_

In [6]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(X, 
                                                        df_heart['AHD'], 
                                                        test_size=0.3,
                                                       stratify = df_heart['AHD'],
                                                       shuffle = True) # 70% training / 30% test

    params,model = antrenament_scikitlearn_svmclsf(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
        'parametrii': params}
        )
    
    
print("Acuratete medie:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Iteration 0


Iteration 1


Iteration 2


Iteration 3


Iteration 4


Acuratete medie: 0.8177777777777777


,accuracy_score,parametrii
0,0.833333,"{'svc__C': 0.5, 'svc__kernel': 'rbf'}"
1,0.788889,"{'svc__C': 1, 'svc__kernel': 'linear'}"
2,0.800000,"{'svc__C': 0.5, 'svc__kernel': 'rbf'}"
3,0.811111,"{'svc__C': 1, 'svc__kernel': 'rbf'}"
4,0.855556,"{'svc__C': 1, 'svc__kernel': 'linear'}"


## Exemplu de antrenare regresor Random Classifier cu GridSearch

In [7]:
from sklearn.ensemble import RandomForestClassifier




In [8]:
def antrenament_scikitlearn_rfc(X,y):
    
    # instațiem un model
    model = RandomForestClassifier()#random_state=42)
    
    # setăm parametrii pe care vrem să îi încercăm
    param_grid = {
                    'n_estimators': [100,1000], #[100, 200, 500, 1000],
                    'max_features': ['sqrt','log2', int(round(X.shape[0]/3,0)) ], #cutuma pt regresie: 1/3 din numarul de atribute
                    'max_samples' : [0.1, 0.2, 0.3, 0.4] ,
                    #'criterion' :['gini', 'entropy']
                 }
    
    # folosim  antrenare cu validare încrucișată
    CV_model = GridSearchCV(estimator=model, 
                          param_grid=param_grid, 
                          scoring = 'accuracy', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 10, 
                          n_jobs = 6,
                         error_score='raise')
    # pasut de antrenare
    CV_model.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_model.best_params_, CV_model.best_estimator_



In [9]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(X, 
                                                        df_heart['AHD'], 
                                                        test_size=0.3,
                                                       stratify = df_heart['AHD'],
                                                       shuffle = True) # 70% training / 30% test

    params,model = antrenament_scikitlearn_rfc(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
        'parametrii': params}
        )
    
    
print("Acuratete medie:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Iteration 0


Iteration 1


Iteration 2


Iteration 3


Iteration 4


Acuratete medie: 0.8333333333333334


,accuracy_score,parametrii
0,0.800000,"{'max_features': 'sqrt', 'max_samples': 0.1, '..."
1,0.833333,"{'max_features': 'sqrt', 'max_samples': 0.1, '..."
2,0.855556,"{'max_features': 'sqrt', 'max_samples': 0.1, '..."
3,0.833333,"{'max_features': 'log2', 'max_samples': 0.1, '..."
4,0.844444,"{'max_features': 69, 'max_samples': 0.2, 'n_es..."


In [10]:
print("Average MAE:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Average MAE: 0.8333333333333334


,accuracy_score,parametrii
0,0.800000,"{'max_features': 'sqrt', 'max_samples': 0.1, '..."
1,0.833333,"{'max_features': 'sqrt', 'max_samples': 0.1, '..."
2,0.855556,"{'max_features': 'sqrt', 'max_samples': 0.1, '..."
3,0.833333,"{'max_features': 'log2', 'max_samples': 0.1, '..."
4,0.844444,"{'max_features': 69, 'max_samples': 0.2, 'n_es..."


# Exemplu de antrenare regresor XGBoost cu Grid Search pentru parametrii

In [11]:
from sklearn.ensemble import GradientBoostingClassifier

In [12]:
def antrenament_scikitlearn_xgb_c(X,y):
    
    model = GradientBoostingClassifier()#random_state=42)
    param_grid = {
                    'n_estimators' : [10, 100, 1000],
                    'learning_rate' : [0.001, 0.01, 0.1],
                    'subsample' : [0.2, 0.6, 0.8],
                    'max_depth' : [3, 7, 9]
                 }
     # folosim  antrenare cu validare încrucișată
    CV_model = GridSearchCV(estimator=model, 
                          param_grid=param_grid, 
                          scoring = 'accuracy', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 10, 
                          n_jobs = 6,
                         error_score='raise')
    # pasut de antrenare
    CV_model.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_model.best_params_, CV_model.best_estimator_



Apelarea functiei de antrenament:

In [13]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(X, 
                                                        df_heart['AHD'], 
                                                        test_size=0.3,
                                                       stratify = df_heart['AHD'],
                                                       shuffle = True) # 70% training / 30% test

    params,model = antrenament_scikitlearn_xgb_c(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
        'parametrii': params}
        )
    
    
print("Acuratete medie:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Iteration 0


Iteration 1


Iteration 2


Iteration 3


Iteration 4


Acuratete medie: 0.7866666666666667


,accuracy_score,parametrii
0,0.800000,"{'learning_rate': 0.001, 'max_depth': 3, 'n_es..."
1,0.811111,"{'learning_rate': 0.1, 'max_depth': 7, 'n_esti..."
2,0.777778,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
3,0.788889,"{'learning_rate': 0.01, 'max_depth': 3, 'n_est..."
4,0.755556,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."


# COMPARARE CLASIFICATORI:
1. spargeti setul de date in X_train,X_test_y_train,y_test =train_test_split(X, 
                                                        df_proiect
                                                        test_size=0.3,
                                                       stratify = df_proiect[<nume variabila tinta>],
                                                       shuffle = True)
    
 2. antrenati 3 tipuri de clasificatori diferiti pe X_train
    
 3. evaluati clasficatorii pe acelasi set de date nefolosit la antrenament, X_test.
    
 

In [14]:

# Split dataset în set de antrenament și set de test 
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    df_heart['AHD'], 
                                                    test_size=0.3,
                                                   stratify = df_heart['AHD'],
                                                   shuffle = True) # 70% training / 30% test

params,model = antrenament_scikitlearn_xgb_c(X_train, y_train)

# prognozăm cu cel mai bun model
y_test_pred = model.predict(X_test)
results_CV_paralel.append({
    'metoda':'xgb',
    'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
    'parametrii': params}
    )
params,model = antrenament_scikitlearn_svmclsf (X_train, y_train)
# prognozăm cu cel mai bun model
y_test_pred = model.predict(X_test)
results_CV_paralel.append({
    'metoda':'svmcls',
    'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
    'parametrii': params}
    )

params,model = antrenament_scikitlearn_rfc (X_train, y_train)
# prognozăm cu cel mai bun model
y_test_pred = model.predict(X_test)
results_CV_paralel.append({
    'metoda':'decision tree classifier',
    'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
    'parametrii': params}
    )

#... random forest
#... logistic regression